In [1]:
import torch
import numpy as np
from torch_geometric.data import Data

def create_random_graph(num_nodes, num_edges=None, num_node_features=10, 
                         num_edge_features=None, with_node_labels=False, 
                         with_graph_label=False, with_pos=False, 
                         edge_probability=0.2, directed=False, self_loops=False):
    """
    Creates a PyTorch Geometric graph with random values.
    
    Parameters:
    -----------
    num_nodes : int
        Number of nodes in the graph
    num_edges : int, optional
        Number of edges in the graph. If None, edges are generated based on edge_probability
    num_node_features : int, optional
        Number of features per node
    num_edge_features : int, optional
        Number of features per edge. If None, no edge features are created
    with_node_labels : bool, optional
        Whether to create random node labels (default: False)
    with_graph_label : bool, optional
        Whether to create a random graph-level label (default: False)
    with_pos : bool, optional
        Whether to create random 2D node positions (default: False)
    edge_probability : float, optional
        Probability of edge creation when num_edges is None (default: 0.2)
    directed : bool, optional
        Whether to create a directed graph (default: False)
    self_loops : bool, optional
        Whether to allow self-loops (default: False)
        
    Returns:
    --------
    data : torch_geometric.data.Data
        PyTorch Geometric graph data object with random values
    """
    # Create random node features
    x = torch.randn(num_nodes, num_node_features)
    
    # Create random edges
    if num_edges is None:
        # Generate random edges based on probability
        possible_edges = num_nodes * num_nodes if self_loops else num_nodes * (num_nodes - 1)
        if not directed:
            possible_edges = possible_edges // 2
        
        num_edges = int(possible_edges * edge_probability)
    
    # Generate random edge_index
    edge_index = None
    if num_edges > 0:
        if directed:
            # For directed graphs
            edge_source = torch.randint(0, num_nodes, (num_edges,))
            edge_target = torch.randint(0, num_nodes, (num_edges,))
            
            # Remove self-loops if not allowed
            if not self_loops:
                mask = edge_source != edge_target
                edge_source = edge_source[mask]
                edge_target = edge_target[mask]
                
                # Re-sample to maintain desired edge count
                while edge_source.size(0) < num_edges:
                    additional = num_edges - edge_source.size(0)
                    new_source = torch.randint(0, num_nodes, (additional,))
                    new_target = torch.randint(0, num_nodes, (additional,))
                    mask = new_source != new_target
                    edge_source = torch.cat([edge_source, new_source[mask]])
                    edge_target = torch.cat([edge_target, new_target[mask]])
                
                # Trim if we got too many edges
                if edge_source.size(0) > num_edges:
                    edge_source = edge_source[:num_edges]
                    edge_target = edge_target[:num_edges]
            
            edge_index = torch.stack([edge_source, edge_target], dim=0)
        else:
            # For undirected graphs
            # Create a set of unique edges (avoiding duplicates)
            edges = set()
            while len(edges) < num_edges:
                i = np.random.randint(0, num_nodes)
                j = np.random.randint(0, num_nodes)
                
                # Skip self-loops if not allowed
                if i == j and not self_loops:
                    continue
                
                # For undirected graphs, ensure (i,j) and (j,i) are treated as the same edge
                if i > j:
                    i, j = j, i
                
                edges.add((i, j))
            
            # Convert to tensor
            edge_list = list(edges)
            source = [e[0] for e in edge_list]
            target = [e[1] for e in edge_list]
            
            # For undirected graphs, we need both (i,j) and (j,i) in edge_index
            if not directed:
                source += [e[1] for e in edge_list]
                target += [e[0] for e in edge_list]
            
            edge_index = torch.tensor([source, target], dtype=torch.long)
    else:
        # Empty graph (no edges)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    
    # Create random edge attributes if requested
    edge_attr = None
    if num_edge_features is not None and num_edge_features > 0:
        num_actual_edges = edge_index.size(1)
        edge_attr = torch.randn(num_actual_edges, num_edge_features)
    
    # Create random node labels if requested
    y = None
    if with_node_labels:
        # Binary classification for nodes (can be modified for multi-class)
        y = torch.randint(0, 2, (num_nodes,))
    elif with_graph_label:
        # Binary classification for the graph (can be modified for multi-class)
        y = torch.randint(0, 2, (1,))
    
    # Create random positions if requested
    pos = None
    if with_pos:
        pos = torch.randn(num_nodes, 2)  # 2D positions
    
    # Create node IDs
    node_ids = torch.arange(num_nodes)
    
    # Create the Data object with the generated random values
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        pos=pos,
        node_id=node_ids
    )
    
    return data

In [3]:
# Example training code for the TemporalGraphPredictor model

import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data, Batch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create a simple dataset for demonstration
def create_dummy_dataset(num_sequences=100, sequence_length=5, num_nodes=10):
    """
    Create a dummy dataset of graph sequences for training.
    
    Args:
        num_sequences: Number of sequences to generate
        sequence_length: Length of each sequence
        num_nodes: Number of nodes in each graph
        
    Returns:
        list: List of graph sequences
    """
    dataset = []
    
    for i in range(num_sequences):
        sequence = []
        
        for t in range(sequence_length):
            # Create node features (x, y, z, other features...)
            x = torch.randn(num_nodes, 10, device=device)  # 10 node features
            
            # Create edges (fully connected for simplicity)
            edge_index = []
            for src in range(num_nodes):
                for dst in range(num_nodes):
                    if src != dst:  # No self-loops
                        edge_index.append([src, dst])
            edge_index = torch.tensor(edge_index, dtype=torch.long, device=device).t()
            
            # Create edge features
            edge_attr = torch.randn(edge_index.size(1), 3, device=device)  # 3 edge features
            
            # Create node IDs
            node_id = torch.arange(num_nodes, device=device)
            
            # Create target values (future positions)
            # Instead of continuous values, we'll create class labels (0-3)
            y = torch.randint(0, 4, (num_nodes,), device=device)  # Class labels
            
            # Create graph
            graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, 
                         y=y, node_id=node_id)
            
            sequence.append(graph)
        
        dataset.append(sequence)
    
    return dataset

# Split dataset into train and validation sets
def train_val_split(dataset, val_ratio=0.2):
    """Split dataset into training and validation sets"""
    val_size = int(len(dataset) * val_ratio)
    train_dataset = dataset[val_size:]
    val_dataset = dataset[:val_size]
    return train_dataset, val_dataset

# Training function
def train_model(model, train_dataset, val_dataset, epochs=50, batch_size=16, lr=0.001):
    """
    Train the TemporalGraphPredictor model
    
    Args:
        model: The model to train
        train_dataset: Training dataset
        val_dataset: Validation dataset
        epochs: Number of training epochs
        batch_size: Batch size
        lr: Learning rate
    
    Returns:
        dict: Training history
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for classification
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_losses = []
        train_correct = 0
        train_total = 0
        
        # Create batches
        indices = np.random.permutation(len(train_dataset))
        
        for i in tqdm(range(0, len(indices), batch_size), desc=f"Epoch {epoch+1}/{epochs}"):
            batch_indices = indices[i:i+batch_size]
            batch_sequences = [train_dataset[idx] for idx in batch_indices]
            
            # Forward pass
            optimizer.zero_grad()
            predictions = model(batch_sequences)  # Shape: [batch_size*num_nodes, 4]
            
            # Get targets from the last graphs in each sequence
            targets = []
            for seq_idx in batch_indices:
                targets.append(train_dataset[seq_idx][-1].y)
            targets = torch.cat(targets, dim=0)  # Shape: [batch_size*num_nodes]
            
            # Compute loss
            loss = criterion(predictions, targets)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Calculate accuracy
            _, predicted = torch.max(predictions.data, 1)  # Get the index of the max log-probability
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()
            
            train_losses.append(loss.item())
        
        avg_train_loss = np.mean(train_losses)
        train_accuracy = 100 * train_correct / train_total
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_accuracy)
        
        # Validation
        model.eval()
        val_losses = []
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for i in range(0, len(val_dataset), batch_size):
                batch_sequences = val_dataset[i:i+batch_size]
                
                # Forward pass
                predictions = model(batch_sequences)
                
                # Get targets
                targets = []
                for seq in batch_sequences:
                    targets.append(seq[-1].y)
                targets = torch.cat(targets, dim=0)
                
                # Compute loss
                loss = criterion(predictions, targets)
                val_losses.append(loss.item())
                
                # Calculate accuracy
                _, predicted = torch.max(predictions.data, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()
        
        avg_val_loss = np.mean(val_losses)
        val_accuracy = 100 * val_correct / val_total
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_accuracy)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")
    
    return history

# Example usage
if __name__ == "__main__":
    # Create dataset
    print("Creating dummy dataset...")
    dataset = create_dummy_dataset(num_sequences=200, sequence_length=5, num_nodes=10)
    train_dataset, val_dataset = train_val_split(dataset)
    
    # Initialize model
    first_graph = train_dataset[0][0]
    model = TemporalGraphPredictor(
        node_features=first_graph.x.size(1),
        hidden_dim=64,
        lstm_hidden_dim=128,
        output_dim=4,  # 4 classes
        edge_dim=first_graph.edge_attr.size(1)
    ).to(device)
    
    # Train model
    print("Training model...")
    history = train_model(model, train_dataset, val_dataset, epochs=20)
    
    # Plot training history
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Save the trained model
    torch.save(model.state_dict(), 'temporal_graph_predictor.pt')
    print("Model saved to 'temporal_graph_predictor.pt'")


Using device: cuda
Creating dummy dataset...


NameError: name 'TemporalGraphPredictor' is not defined

In [47]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data, Batch
import numpy as np

class TemporalGraphPredictor(nn.Module):
    """
    Model for predicting node labels from a sequence of temporal graphs.
    Uses GNN for spatial features and LSTM for temporal dynamics.
    """
    def __init__(self, node_features, hidden_dim, lstm_hidden_dim, output_dim, edge_dim=None):
        super(TemporalGraphPredictor, self).__init__()
        
        # GNN for processing each graph
        self.gnn_encoder = nn.Sequential(
            GCNConv(node_features, hidden_dim),
            nn.ReLU(),
            GCNConv(hidden_dim, hidden_dim)
        )
        
        # LSTM for processing the sequence of node embeddings
        self.lstm = nn.LSTM(hidden_dim, lstm_hidden_dim, batch_first=True)
        
        # Final prediction layer
        self.predictor = nn.Linear(lstm_hidden_dim, output_dim)
        
    def forward(self, graph_sequence):
        """
        Process a sequence of graphs and predict node labels for the last graph.
        
        Args:
            graph_sequence: List of graph objects [graph_1, graph_2, ..., graph_t]
            
        Returns:
            Predictions for each node in the last graph
        """
        batch_size = len(graph_sequence)
        sequence_length = len(graph_sequence[0])
        
        # Process each graph in the sequence for each batch
        all_node_embeddings = []
        all_node_ids = []
        
        for batch_idx in range(batch_size):
            seq_embeddings = []
            seq_node_ids = []
            
            for t in range(sequence_length):
                graph = graph_sequence[batch_idx][t]
                # Apply GNN layers manually instead of using Sequential
                x = graph.x
                x = self.gnn_encoder[0](x, graph.edge_index)
                x = self.gnn_encoder[1](x)  # ReLU
                x = self.gnn_encoder[2](x, graph.edge_index)
                
                seq_embeddings.append(x)
                seq_node_ids.append(graph.node_id)
            
            all_node_embeddings.append(seq_embeddings)
            all_node_ids.append(seq_node_ids)
        
        # Process the last graph in each sequence
        predictions = []
        
        for batch_idx in range(batch_size):
            # Get node embeddings for this sequence
            node_embs_sequence = all_node_embeddings[batch_idx]  # [seq_len, num_nodes, hidden_dim]
            node_ids_sequence = all_node_ids[batch_idx]  # [seq_len, num_nodes]
            
            # Create a mapping of node IDs to their temporal embeddings
            node_temporal_embs = {}
            
            # For each unique node ID in the last graph
            last_graph_node_ids = node_ids_sequence[-1]
            
            for node_idx, node_id in enumerate(last_graph_node_ids):
                # Collect embeddings for this node across time
                temporal_embs = []
                
                for t in range(sequence_length):
                    # Check if this node exists in graph at time t
                    if node_id in node_ids_sequence[t]:
                        # Find position of this node in the graph at time t
                        t_node_idx = node_ids_sequence[t].index(node_id)
                        temporal_embs.append(node_embs_sequence[t][t_node_idx])
                    else:
                        # If node doesn't exist at this time, use zero embedding
                        temporal_embs.append(torch.zeros_like(node_embs_sequence[0][0]))
                
                # Stack temporal embeddings for this node
                node_temporal_embs[node_id] = torch.stack(temporal_embs).unsqueeze(0)  # [1, seq_len, hidden_dim]
            
            # Process each node's temporal embeddings through LSTM
            node_predictions = {}
            
            for node_id, temporal_emb in node_temporal_embs.items():
                lstm_out, _ = self.lstm(temporal_emb)
                # Take the last output from LSTM
                last_output = lstm_out[:, -1, :]
                # Make prediction
                pred = self.predictor(last_output)
                node_predictions[node_id] = pred.squeeze(0)
            
            # Arrange predictions in the same order as nodes in the last graph
            batch_preds = []
            for node_id in last_graph_node_ids:
                batch_preds.append(node_predictions[node_id])
            
            predictions.append(torch.stack(batch_preds))
        
        return predictions

def create_random_graph(num_nodes, num_node_features):
    """
    Create a random graph with node IDs
    """
    # Create random node features
    x = torch.randn(num_nodes, num_node_features)
    
    # Create random edge connections
    edge_index = []
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j and torch.rand(1).item() > 0.7:  # 30% chance of edge
                edge_index.append([i, j])
    
    if not edge_index:  # Ensure at least one edge
        edge_index = [[0, 1]]
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    # Create random edge attributes
    edge_attr = torch.randn(edge_index.size(1), 2)  # 2 features per edge
    
    # Create random node IDs
    node_id = list(range(1000, 1000 + num_nodes))
    
    # Create the graph
    graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    graph.node_id = node_id
    
    return graph

def create_dummy_dataset(num_sequences, sequence_length, num_nodes):
    """
    Create a dummy dataset of temporal graph sequences
    """
    dataset = []
    
    for _ in range(num_sequences):
        # Create a sequence of graphs
        sequence = []
        
        # Start with a base set of node IDs
        base_node_ids = list(range(1000, 1000 + num_nodes))
        
        for t in range(sequence_length):
            # Determine how many nodes to use in this graph (allow for some variation)
            current_num_nodes = max(3, num_nodes + np.random.randint(-2, 3))
            
            # Select node IDs for this graph (with some consistency between timesteps)
            if t == 0:
                current_node_ids = base_node_ids[:current_num_nodes]
            else:
                # Keep some nodes from previous graph, add/remove some
                prev_node_ids = sequence[-1].node_id
                keep_prob = 0.8
                kept_nodes = [node_id for node_id in prev_node_ids if np.random.random() < keep_prob]
                
                # Add some new nodes if needed
                available_new_nodes = [nid for nid in base_node_ids if nid not in kept_nodes]
                num_new_nodes = current_num_nodes - len(kept_nodes)
                
                if num_new_nodes > 0 and available_new_nodes:
                    new_nodes = np.random.choice(
                        available_new_nodes, 
                        size=min(num_new_nodes, len(available_new_nodes)), 
                        replace=False
                    ).tolist()
                    current_node_ids = kept_nodes + new_nodes
                else:
                    current_node_ids = kept_nodes
            
            # Create graph with these node IDs
            graph = create_random_graph(len(current_node_ids), num_node_features=13)
            graph.node_id = current_node_ids
            
            # Add random node labels for the last graph in sequence
            if t == sequence_length - 1:
                graph.y = torch.randint(0, 4, (len(current_node_ids),))  # 4 classes
            
            sequence.append(graph)
        
        dataset.append(sequence)
    
    return dataset

def train_val_split(dataset, val_ratio=0.2):
    """Split dataset into training and validation sets"""
    val_size = int(len(dataset) * val_ratio)
    train_dataset = dataset[val_size:]
    val_dataset = dataset[:val_size]
    return train_dataset, val_dataset

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
import pickle
import os
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

def train_model(model, train_dataset, val_dataset, epochs=50, lr=0.001, save_path='model.pkl'):
    """
    Train the TemporalGraphPredictor model and save it to a file.
    
    Args:
        model: The TemporalGraphPredictor model
        train_dataset: Training dataset
        val_dataset: Validation dataset
        epochs: Number of training epochs
        lr: Learning rate
        save_path: Path to save the trained model
    
    Returns:
        Trained model and training history
    """
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = CrossEntropyLoss()
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': []
    }
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for batch in tqdm(train_dataset, desc=f"Epoch {epoch+1}/{epochs} (Training)"):
            optimizer.zero_grad()
            
            # Forward pass
            predictions = model(batch)
            
            # Calculate loss for each graph in the batch
            batch_loss = 0.0
            for batch_idx, pred in enumerate(predictions):
                # Get the last graph in the sequence
                target = batch[batch_idx][-1].y.to(device)
                batch_loss += criterion(pred, target)
            
            batch_loss /= len(predictions)
            
            # Backward pass
            batch_loss.backward()
            optimizer.step()
            
            train_loss += batch_loss.item()
        
        train_loss /= len(train_dataset)
        history['train_loss'].append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in tqdm(val_dataset, desc=f"Epoch {epoch+1}/{epochs} (Validation)"):
                # Forward pass
                predictions = model(batch)
                
                # Calculate loss and accuracy for each graph in the batch
                batch_loss = 0.0
                for batch_idx, pred in enumerate(predictions):
                    # Get the last graph in the sequence
                    target = batch[batch_idx][-1].y.to(device)
                    batch_loss += criterion(pred, target)
                    
                    # Calculate accuracy
                    _, predicted = torch.max(pred, 1)
                    total += target.size(0)
                    correct += (predicted == target).sum().item()
                
                batch_loss /= len(predictions)
                val_loss += batch_loss.item()
        
        val_loss /= len(val_dataset)
        val_accuracy = correct / total
        
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, "
              f"Val Accuracy: {val_accuracy:.4f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            # Save the model
            with open(save_path, 'wb') as f:
                pickle.dump(model, f)
            print(f"Model saved to {save_path}")
    
    return model, history

# Generate synthetic dataset
print("Generating synthetic dataset...")
train_dataset, val_dataset = train_val_split(dataset)

# Initialize model
first_graph = train_dataset[0][0]
model = TemporalGraphPredictor(
    node_features=first_graph.x.size(1),
    hidden_dim=64,
    lstm_hidden_dim=128,
    output_dim=4,  # 4 classes
    edge_dim=first_graph.edge_attr.size(1)
).to(device)

# Train model
print("Training model...")
trained_model, history = train_model(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    epochs=20,
    lr=0.001,
    save_path='models/temporal_graph_predictor.pkl'
)

print("Training complete!")
